## Example usage

In [ ]:
%load_ext autoreload
%autoreload 2

# Run in parent dir
import os
from pathlib import Path
import torch

from dual_ifm.classification.models import CNNwithProjectorandClassifier
from dual_ifm.tsimcne.models import CNNwithProjector
from dual_ifm.utils import datasets

# Run in parent dir
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

device = 'cuda:0'

In [ ]:
# Load dataset
dataset_name = 'idrid'
feature_name = 'dr'
image_size = (256, 256)

normalization = {'mean': datasets.NORMALIZATION_MEAN[dataset_name], 'sd': datasets.NORMALIZATION_SD[dataset_name]}
transform = datasets.get_augmentations(img_size=image_size, normalization=normalization, imagenet=False)['test']
dataset, mapping = datasets.load_dataset(dataset_name=dataset_name, transform=transform, image_size=image_size, feature_name=feature_name, drop_nan=True, sample_size=None, split='all')
n_classes = len(mapping)

### 1. Usage of the pretrained model
After you have downloaded the weights you can load any of the released checkpoints.

In [ ]:
checkpoint_file = 'path/to/model/weights'
backbone = 'bagnet33' # options: bagnet33, resnet50

model = CNNwithProjector(img_size=image_size, backbone=backbone)

# Load checkpoint if it exists
if os.path.exists(checkpoint_file):
    checkpoint = torch.load(checkpoint_file, map_location=torch.device('cpu'), weights_only=False)

    # Necessary for tsimcne pretraining, comment to load simclr weights
    model.mutate_projector()

    model.load_state_dict(checkpoint['state_dict'])
else:
    print('Checkpoint file does not exist.')

model.to(device);

In [ ]:
image, label = dataset[0]
image = image.unsqueeze(0).to(device)

# h is the high dimensional 2048D embedding and z to 2D embedding
with torch.no_grad():
    h, z = model(image)

### 2. Usage of a finetuned model

In [ ]:
checkpoint_file = 'path/to/model/weights'

# Load checkpoint if it exists
if os.path.exists(checkpoint_file):
    checkpoint = torch.load(checkpoint_file, map_location=torch.device('cpu'), weights_only=False)
    backbone = checkpoint['backbone']
    model = CNNwithProjectorandClassifier(img_size=image_size, backbone=backbone, n_classes=n_classes)
    model.load_state_dict(checkpoint['state_dict'])

model.to(device)

In [ ]:
image, label = dataset[0]
image = image.unsqueeze(0).to(device)

# z is the 2D embedding if the backbone was pretrained with tsimcne otherwise it is h
with torch.no_grad():
    z, y = model(image)

In [ ]:
# To get the local evidence map
with torch.no_grad():
    heatmap = model(image, return_heatmap=True)

Check the other files in example_scripts for ways to finetune, train a linear probe and align the 2D projector after finetuning, and even pretraining Dual-IFM from scratch. In all cases, you will have to modify the utils/datasets.py script to specify how to load your own local copies of datasets.

Examples on how to visualize global and local interpretation can be found in the notebooks in the interpretation folder. You can use the eval scripts in the finetune, tsimcne and simclr directories, depending on the model you want to use, to compute 2D embeddings and store them in a json file to later load and visualize.